# MLP Weight Spectrum

Spectral analysis of MLP weight matrices: singular value spectra,
effective rank, input-output alignment, and neuron norm distribution.

## Why This Matters

MLP weight spectrum provides tools for understanding how feed-forward layers transform and store information. Since MLPs have been shown to function as key-value memories, understanding their internal mechanics is essential for locating knowledge and understanding factual recall.

**Key references:**
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores
- [Bills et al. (2023) "Language Models Can Explain Neurons"](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html) — Automated neuron interpretation methods

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.mlp_weight_spectrum import (
    mlp_input_weight_spectrum, mlp_output_weight_spectrum,
    mlp_in_out_alignment, mlp_neuron_norm_distribution,
    mlp_weight_spectrum_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Input Weight Spectrum (W_in)

Singular values of the input projection matrix.

In [ ]:
for layer in range(4):
    result = mlp_input_weight_spectrum(model, layer=layer)
    svs = ', '.join(f'{s:.4f}' for s in result['top_singular_values'][:5])
    print(f"  Layer {layer}: rank={result['effective_rank']:.2f}, "
          f"cond={result['condition_number']:.1f}")
    print(f"    Top SVs: [{svs}]")

## Output Weight Spectrum (W_out)

In [ ]:
for layer in range(4):
    result = mlp_output_weight_spectrum(model, layer=layer)
    print(f"  Layer {layer}: rank={result['effective_rank']:.2f}, "
          f"energy={result['total_energy']:.4f}")

## Input-Output Alignment

Properties of the W_in @ W_out product (residual-to-residual mapping).

In [ ]:
for layer in range(4):
    result = mlp_in_out_alignment(model, layer=layer)
    print(f"  Layer {layer}: rank={result['product_effective_rank']:.2f}, "
          f"trace={result['product_trace']:.4f}, frob={result['product_frobenius']:.4f}")

## Neuron Norm Distribution

In [ ]:
for layer in range(4):
    result = mlp_neuron_norm_distribution(model, layer=layer)
    print(f"  Layer {layer}: in_norm={result['in_mean_norm']:.4f}±{result['in_std_norm']:.4f}, "
          f"out_norm={result['out_mean_norm']:.4f}±{result['out_std_norm']:.4f}, "
          f"corr={result['in_out_correlation']:.4f}")

## Weight Spectrum Summary

In [ ]:
result = mlp_weight_spectrum_summary(model)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: in_rank={p['in_rank']:.2f}, "
          f"out_rank={p['out_rank']:.2f}")